# Retail Cash Flows — Nasdaq 100 Sector Aggregation - Example Script | REST API

This notebook demonstrates how to:

1. Retrieve all constituents of a benchmark aggregate
2. Pull daily retail cash flows for each security
3. Retrieve sector metadata
4. Aggregate retail flows by sector
5. Export security-level and sector-level datasets

This example uses:
- `/aggregates/reference`
- `/series/timeseries`
- `/series/securities/{id}/summary` 

# Authentication

## Login Using Email & Password

In [ ]:
import os
import requests
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from vanda import VandaClient


VANDA_EMAIL = "VANDA_EMAIL" # <-- Replace with your email
VANDA_PASSWORD = "VANDA_PASSWORD"   # <-- Replace with your password

if VANDA_EMAIL: os.environ["VANDA_EMAIL"] = VANDA_EMAIL

if VANDA_PASSWORD: os.environ["VANDA_PASSWORD"] = VANDA_PASSWORD


API_BASE = "https://api.vanda-analytics.com"
TEST_TOKEN_URL = f"{API_BASE}/series/test/token"

email = VANDA_EMAIL or os.getenv("VANDA_EMAIL")
password = VANDA_PASSWORD or os.getenv("VANDA_PASSWORD")

if not email or not password:
    raise ValueError(
        "Please either:\n"
        "1) Set VANDA_EMAIL and VANDA_PASSWORD in the config cell, OR\n"
        "2) Set them as environment variables."
    )

response = requests.post(
    TEST_TOKEN_URL,
    json={"email": email, "password": password}
)

response.raise_for_status()

TOKEN = response.json()["access_token"]

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json"
}

print("Authentication successful.")

client = VandaClient(
    token=TOKEN,
    base_url=API_BASE
)

Authentication successful.


## Configuration

We will pull:

- Nasdaq 100 constituents
- Daily retail cash flows
- Full history from 2020 onward

In [2]:
VANDA_ID = "VNDA2100003"   # Nasdaq 100
START_DATE = "2020-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

FIELDS = [
    "retail_buy_turnover",
    "retail_sell_turnover",
    "retail_net_turnover",
]

SERIES_URL = f"{API_BASE}/series/timeseries"
AGG_REF_URL = f"{API_BASE}/aggregates/reference"
SUMMARY_URL = f"{API_BASE}/series/securities"

## Step 1 — Retrieve Benchmark Constituents

In [3]:
r = requests.get(
    f"{AGG_REF_URL}/{VANDA_ID}/constituents",
    headers=HEADERS
)

r.raise_for_status()
constituent_ids = r.json()["constituent_ids"]

print(f"Total constituents: {len(constituent_ids)}")

vanda_ids = [f"VNDA{sid}" for sid in constituent_ids]

Total constituents: 101


## Step 2 — Retrieve Retail Cash Flows for Each Security

In [4]:
all_data = []

for vid in tqdm(vanda_ids):

    params = {
        "vanda_id": vid,
        "interval": "1d",
        "start_date": START_DATE,
        "end_date": END_DATE,
        "asset_class": "cash",
        "fields": FIELDS,
        "records_per_page": 5000,
        "page_number": 1,
        "order": "asc"
    }

    r = requests.get(SERIES_URL, headers=HEADERS, params=params)

    if r.status_code != 200:
        continue

    payload = r.json()

    if payload.get("status") != "success":
        continue

    results = payload.get("results", [])
    if not results:
        continue

    df_ts = pd.DataFrame(results)
    all_data.append(df_ts)

if not all_data:
    raise ValueError("All series returned empty.")

df_all = pd.concat(all_data, ignore_index=True)
df_all["date"] = pd.to_datetime(df_all["ts"])

print("Rows downloaded:", len(df_all))
df_all.columns

100%|██████████| 101/101 [02:08<00:00,  1.27s/it]


Rows downloaded: 151550


Index(['ts', 'vanda_id', 'symbol', 'name', 'sector', 'industry', 'asset_type',
       'is_active', 'updated_time', 'asset_class', 'retail_buy_turnover',
       'retail_sell_turnover', 'retail_net_turnover', 'retail_total_turnover',
       'moneyness', 'size', 'call_turnover', 'put_turnover', 'cp_turnover',
       'net_cp_turnover', 'call_notional', 'put_notional', 'cp_notional',
       'net_cp_notional', 'px', 'px_chg', 'z_score', 'momentum', 'ris', 'rank',
       'date'],
      dtype='object')

## Step 3 — Retrieve Sector Mapping

In [6]:
meta_rows = []

unique_ids = df_all["vanda_id"].unique()

for vid in tqdm(unique_ids):

    r = requests.get(
        f"{SUMMARY_URL}/{vid}/summary",
        headers=HEADERS,
        params={
            "interval": "1d",
            "asset_class": "cash"
        }
    )

    if r.status_code != 200:
        continue

    data = r.json()

    meta_rows.append({
        "vanda_id": vid,
        "ticker": data.get("ticker"),
        "sector": data.get("sector")
    })

df_meta = pd.DataFrame(meta_rows)

print("Sector mappings pulled:", len(df_meta))
df_meta.head()

100%|██████████| 101/101 [01:22<00:00,  1.23it/s]

Sector mappings pulled: 101


,vanda_id,ticker,sector
0,VNDA1000013,AAPL,Information Technology
1,VNDA1000098,ADBE,Information Technology
2,VNDA1000105,ADI,Information Technology
3,VNDA1000112,ADP,Industrials
4,VNDA1000118,ADSK,Information Technology


## Step 4 — Merge Sector Information

In [7]:
if "sector" in df_all.columns:
    df_all = df_all.drop(columns=["sector"])

df_all = df_all.merge(
    df_meta[["vanda_id", "sector"]],
    on="vanda_id",
    how="left"
)

## Step 5 — Aggregate by Sector

In [8]:
df_sector = (
    df_all
    .groupby(["date", "sector"])[
        ["retail_buy_turnover",
         "retail_sell_turnover",
         "retail_net_turnover"]
    ]
    .sum()
    .reset_index()
)

df_sector.head()

,date,sector,retail_buy_turnover,retail_sell_turnover,retail_net_turnover
0,2020-01-02 00:00:00+00:00,Communication Services,1.973968e+08,1.990956e+08,-1698741.68
1,2020-01-02 00:00:00+00:00,Consumer Discretionary,3.488165e+08,3.329344e+08,15882052.97
2,2020-01-02 00:00:00+00:00,Consumer Staples,5.316455e+07,5.340774e+07,-243187.42
3,2020-01-02 00:00:00+00:00,Energy,4.045573e+06,3.988785e+06,56787.66
4,2020-01-02 00:00:00+00:00,Financials,1.698455e+07,1.835023e+07,-1365679.96


## Step 6 — Export Outputs

In [9]:
df_all.to_csv("r1000_security_retail_cash.csv", index=False)
df_sector.to_csv("r1000_retail_by_sector.csv", index=False)

print("Export complete.")

Export complete.
